# Fase 17A — Congelamento oficial dos três datasets

Esta fase inicia a campanha `THESIS_OFFICIAL_CAMPAIGN_V2`. Ela **não treina modelos**. Seu objetivo é inventariar 100% dos arquivos, fixar a identidade canônica dos datasets, localizar artefatos derivados e splits, criar a árvore oficial de resultados e bloquear a campanha se faltar evidência científica.

Datasets oficiais: PhysioNet/Computing in Cardiology Challenge 2012, Dahl Rats e CheXchoNet. O nome legado `mimic_iii` é preservado somente como rastreabilidade histórica.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, json, os, platform, re, sys, zipfile

CAMPAIGN_ID = 'THESIS_OFFICIAL_CAMPAIGN_V2_20260901'
PROJECT_ROOT = Path('/content/drive/MyDrive/Mestrado_Criptografia')
DATASETS_ROOT = PROJECT_ROOT / 'datasets'
CAMPAIGN_ROOT = PROJECT_ROOT / 'OFFICIAL_CAMPAIGN_V2' / CAMPAIGN_ID
SEED = 42
NUM_CLIENTS = 5
OFFICIAL_ROUNDS = 30
SPLIT_RATIOS = {'train': 0.70, 'validation': 0.15, 'test': 0.15}

DATASETS = {
  'PHYSIONET_CHALLENGE_2012': {
    'canonical_name': 'PhysioNet/Computing in Cardiology Challenge 2012',
    'legacy_labels': ['mimic_iii'],
    'candidate_roots': [
      DATASETS_ROOT / 'prepared' / 'mimic_iii',
      DATASETS_ROOT / 'prepared' / 'physionet_challenge_2012',
      DATASETS_ROOT / 'physionet_challenge_2012'
    ],
    'anti_leakage_unit': 'RecordID/patient',
  },
  'DAHL_RATS': {
    'canonical_name': 'Blood Pressure in Salt-Sensitive Dahl Rats',
    'legacy_labels': [],
    'candidate_roots': [DATASETS_ROOT / 'prepared' / 'dahl_rats'],
    'anti_leakage_unit': 'animal',
  },
  'CHEXCHONET': {
    'canonical_name': 'CheXchoNet',
    'legacy_labels': [],
    'candidate_roots': [
      DATASETS_ROOT / 'chexchonet_official_drop',
      DATASETS_ROOT / 'prepared' / 'chexchonet',
      DATASETS_ROOT / 'chexchonet'
    ],
    'anti_leakage_unit': 'patient',
  },
}
print('Campanha:', CAMPAIGN_ID)
print('Raiz:', CAMPAIGN_ROOT)

In [ ]:
# Cria a estrutura oficial antes de qualquer processamento.
folders = [
  '00_CAMPAIGN_CONTROL', '01_DATASET_FREEZE', '02_PREFLIGHT',
  '03_SMOKE_TESTS', '04_OFFICIAL_RUNS', '05_COMPARISONS', '06_THESIS_EXPORT'
]
for folder in folders:
    (CAMPAIGN_ROOT / folder).mkdir(parents=True, exist_ok=True)
for dataset in DATASETS:
    (CAMPAIGN_ROOT / '01_DATASET_FREEZE' / dataset).mkdir(parents=True, exist_ok=True)
    for scenario in ['BASELINE', 'CKKS', 'HYBRID']:
        (CAMPAIGN_ROOT / '03_SMOKE_TESTS' / dataset / scenario).mkdir(parents=True, exist_ok=True)
        (CAMPAIGN_ROOT / '04_OFFICIAL_RUNS' / dataset / scenario).mkdir(parents=True, exist_ok=True)

campaign_status = {
  'campaign_id': CAMPAIGN_ID,
  'created_at_utc': datetime.now(timezone.utc).isoformat(),
  'status': 'DATASET_FREEZE_RUNNING',
  'usable_in_thesis': False,
  'seed': SEED, 'num_clients': NUM_CLIENTS, 'official_rounds': OFFICIAL_ROUNDS,
  'split_ratios': SPLIT_RATIOS,
  'official_matrix': [f'{d}__{s}' for d in DATASETS for s in ['BASELINE','CKKS','HYBRID']],
}
status_path = CAMPAIGN_ROOT / '00_CAMPAIGN_CONTROL' / 'CAMPAIGN_STATUS.json'
status_path.write_text(json.dumps(campaign_status, indent=2, ensure_ascii=False), encoding='utf-8')
print('Estrutura oficial criada:', CAMPAIGN_ROOT)

In [ ]:
# Resolve uma única raiz por dataset; raízes ambíguas bloqueiam a campanha.
resolved_roots, root_errors = {}, {}
for dataset, cfg in DATASETS.items():
    existing = [p for p in cfg['candidate_roots'] if p.exists()]
    nonempty = [p for p in existing if p.is_file() or any(p.iterdir())]
    if len(nonempty) == 1:
        resolved_roots[dataset] = nonempty[0]
    elif len(nonempty) == 0:
        root_errors[dataset] = 'Nenhuma raiz candidata encontrada'
    else:
        root_errors[dataset] = 'Mais de uma raiz candidata encontrada: ' + ' | '.join(map(str, nonempty))

print(json.dumps({'resolved_roots': {k:str(v) for k,v in resolved_roots.items()}, 'errors': root_errors}, indent=2, ensure_ascii=False))

In [ ]:
# Inventário físico de 100% dos arquivos. SHA-256 lê integralmente cada arquivo.
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with path.open('rb') as fh:
        while True:
            chunk = fh.read(chunk_size)
            if not chunk: break
            h.update(chunk)
    return h.hexdigest()

def preliminary_role(path):
    n = path.name.lower()
    if n.startswith('outcomes-'): return 'LABEL_CANDIDATE'
    if path.suffix.lower() in {'.hea','.dat','.csv','.tsv','.txt','.mat','.npy','.npz','.jpg','.jpeg','.png','.dcm'}: return 'DATA_OR_METADATA_CANDIDATE'
    if path.suffix.lower() in {'.zip','.tar','.gz','.tgz'}: return 'ARCHIVE_CANDIDATE'
    if path.suffix.lower() in {'.md','.pdf','.html','.htm'}: return 'DOCUMENTATION_CANDIDATE'
    if path.suffix.lower() in {'.py','.m','.r','.java','.c','.cpp'}: return 'SOURCE_CODE_CANDIDATE'
    return 'UNCLASSIFIED'

inventory_summaries = {}
for dataset, root in resolved_roots.items():
    outdir = CAMPAIGN_ROOT / '01_DATASET_FREEZE' / dataset
    files = [root] if root.is_file() else sorted(p for p in root.rglob('*') if p.is_file())
    rows, read_errors = [], []
    for i, path in enumerate(files, 1):
        try:
            digest = sha256_file(path)
            status, error = 'READ_OK', ''
        except Exception as exc:
            digest, status, error = '', 'READ_ERROR', repr(exc)
            read_errors.append({'path': str(path), 'error': error})
        rows.append({
          'dataset': dataset, 'absolute_path': str(path),
          'relative_path': path.name if root.is_file() else str(path.relative_to(root)),
          'size_bytes': path.stat().st_size, 'extension': path.suffix.lower(),
          'sha256': digest, 'read_status': status, 'preliminary_role': preliminary_role(path),
          'final_usage_status': 'PENDING_SCIENTIFIC_CLASSIFICATION', 'exclusion_reason': ''
        })
        if i % 500 == 0: print(dataset, 'arquivos processados:', i, '/', len(files))
    csv_path = outdir / 'FILE_INVENTORY_SHA256.csv'
    if rows:
        with csv_path.open('w', newline='', encoding='utf-8') as fh:
            w = csv.DictWriter(fh, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
    summary = {
      'dataset': dataset, 'canonical_name': DATASETS[dataset]['canonical_name'],
      'legacy_labels': DATASETS[dataset]['legacy_labels'], 'resolved_root': str(root),
      'files_discovered': len(files), 'files_read_ok': sum(r['read_status']=='READ_OK' for r in rows),
      'files_read_error': len(read_errors), 'bytes_read': sum(r['size_bytes'] for r in rows if r['read_status']=='READ_OK'),
      'physical_inventory_complete': bool(files) and not read_errors, 'read_errors': read_errors
    }
    (outdir / 'PHYSICAL_INVENTORY_SUMMARY.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
    inventory_summaries[dataset] = summary
print(json.dumps(inventory_summaries, indent=2, ensure_ascii=False))

In [ ]:
# Inventaria também 100% dos membros internos de arquivos ZIP sem confundi-los com arquivos físicos.
archive_summaries = {}
for dataset, root in resolved_roots.items():
    outdir = CAMPAIGN_ROOT / '01_DATASET_FREEZE' / dataset
    physical = [root] if root.is_file() else [p for p in root.rglob('*') if p.is_file()]
    zip_paths = [p for p in physical if p.suffix.lower() == '.zip']
    rows, errors = [], []
    for zpath in zip_paths:
        try:
            with zipfile.ZipFile(zpath) as zf:
                bad = zf.testzip()
                for info in zf.infolist():
                    if not info.is_dir():
                        rows.append({'archive': str(zpath), 'member': info.filename, 'size_bytes': info.file_size, 'compressed_bytes': info.compress_size, 'crc32': f'{info.CRC:08x}'})
                if bad: errors.append({'archive': str(zpath), 'first_bad_member': bad})
        except Exception as exc:
            errors.append({'archive': str(zpath), 'error': repr(exc)})
    if rows:
        with (outdir / 'ARCHIVE_MEMBER_INVENTORY.csv').open('w', newline='', encoding='utf-8') as fh:
            w = csv.DictWriter(fh, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
    archive_summaries[dataset] = {'zip_files': len(zip_paths), 'members': len(rows), 'errors': errors, 'archives_valid': not errors}
    (outdir / 'ARCHIVE_INVENTORY_SUMMARY.json').write_text(json.dumps(archive_summaries[dataset], indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(archive_summaries, indent=2, ensure_ascii=False))

In [ ]:
# Localiza evidências existentes de features, alvos, splits e partições; não cria splits silenciosamente.
SEARCH_ROOTS = [PROJECT_ROOT / 'results', PROJECT_ROOT / 'OFFICIAL_CAMPAIGN_V1', DATASETS_ROOT]
patterns = {
 'feature': re.compile(r'feature|embedding|derived', re.I),
 'target': re.compile(r'target|label|outcome', re.I),
 'split': re.compile(r'split|train|validation|val_|test|partition|manifest', re.I),
 'model': re.compile(r'model|state|checkpoint|config', re.I),
}
evidence = {d:{k:[] for k in patterns} for d in DATASETS}
dataset_tokens = {
 'PHYSIONET_CHALLENGE_2012': re.compile(r'physionet|challenge2012|challenge_2012|mimic', re.I),
 'DAHL_RATS': re.compile(r'dahl|rat|blood.pressure', re.I),
 'CHEXCHONET': re.compile(r'chex', re.I),
}
for base in SEARCH_ROOTS:
    if not base.exists(): continue
    for p in base.rglob('*'):
        if not p.is_file(): continue
        s = str(p)
        for dataset, token in dataset_tokens.items():
            if token.search(s):
                for kind, pat in patterns.items():
                    if pat.search(p.name): evidence[dataset][kind].append(s)
for dataset in evidence:
    for kind in evidence[dataset]: evidence[dataset][kind] = sorted(set(evidence[dataset][kind]))
    out = CAMPAIGN_ROOT / '01_DATASET_FREEZE' / dataset / 'EXISTING_EVIDENCE_CANDIDATES.json'
    out.write_text(json.dumps(evidence[dataset], indent=2, ensure_ascii=False), encoding='utf-8')
print({d:{k:len(v) for k,v in kinds.items()} for d,kinds in evidence.items()})

In [ ]:
# Gate final da Fase 17A. A aprovação física não equivale ainda à aprovação científica.
dataset_gates = {}
for dataset in DATASETS:
    root_ok = dataset in resolved_roots
    inv = inventory_summaries.get(dataset, {})
    archive = archive_summaries.get(dataset, {})
    gate = {
      'dataset': dataset, 'canonical_name': DATASETS[dataset]['canonical_name'],
      'root_resolved': root_ok,
      'physical_inventory_complete': bool(inv.get('physical_inventory_complete')),
      'archives_valid': bool(archive.get('archives_valid', True)),
      'anti_leakage_unit_required': DATASETS[dataset]['anti_leakage_unit'],
      'feature_evidence_candidates': len(evidence[dataset]['feature']),
      'target_evidence_candidates': len(evidence[dataset]['target']),
      'split_evidence_candidates': len(evidence[dataset]['split']),
      'scientific_classification_complete': False,
      'train_validation_test_verified': False,
      'five_client_non_iid_partition_verified': False,
      'approved_for_training': False,
      'next_phase': 'FASE_17B_SCIENTIFIC_SCHEMA_AND_SPLIT_FREEZE'
    }
    dataset_gates[dataset] = gate
    (CAMPAIGN_ROOT / '01_DATASET_FREEZE' / dataset / 'PHASE17A_GATE.json').write_text(json.dumps(gate, indent=2, ensure_ascii=False), encoding='utf-8')

phase_gate = {
 'phase': '17A', 'campaign_id': CAMPAIGN_ID,
 'physical_audit_passed': all(g['root_resolved'] and g['physical_inventory_complete'] and g['archives_valid'] for g in dataset_gates.values()),
 'training_authorized': False,
 'reason': 'O treinamento só será autorizado após schema, alvo, split sem vazamento e partições dos 5 clientes serem congelados na Fase 17B.',
 'dataset_gates': dataset_gates,
}
(CAMPAIGN_ROOT / '00_CAMPAIGN_CONTROL' / 'PHASE17A_MASTER_GATE.json').write_text(json.dumps(phase_gate, indent=2, ensure_ascii=False), encoding='utf-8')
campaign_status['status'] = 'PHASE17A_COMPLETED' if phase_gate['physical_audit_passed'] else 'PHASE17A_BLOCKED'
campaign_status['phase17a_physical_audit_passed'] = phase_gate['physical_audit_passed']
status_path.write_text(json.dumps(campaign_status, indent=2, ensure_ascii=False), encoding='utf-8')
print('='*100)
print(json.dumps(phase_gate, indent=2, ensure_ascii=False))
print('='*100)
print('TRAINING_AUTHORIZED=false — execute a Fase 17B após revisar este gate.')

## Resultado esperado

Ao final, copie para a conversa a saída de `PHASE17A_MASTER_GATE.json` ou compacte a pasta `00_CAMPAIGN_CONTROL` junto com as três pastas de `01_DATASET_FREEZE`. A próxima fase validará o significado científico dos arquivos, congelará treino/validação/teste e as partições não-IID dos cinco clientes.